# Q7: KL Divergence vs Cross-Entropy
**Topic:** Coding-question | **Difficulty:** Medium | **Time:** 20 min
**Sheet:** Grind75ML

---

## Question
Implement KL Divergence and Cross-Entropy from scratch. Compare them and show the mathematical relationship between the two.

## Theory

**Cross-Entropy** measures how well a predicted distribution $Q$ approximates a true distribution $P$:

$$H(P, Q) = -\sum_{x} P(x) \log Q(x)$$

**KL Divergence** measures how much $Q$ diverges from $P$:

$$D_{KL}(P \| Q) = \sum_{x} P(x) \log \frac{P(x)}{Q(x)}$$

**Key Relationship:**

$$H(P, Q) = H(P) + D_{KL}(P \| Q)$$

Where $H(P)$ is the entropy of $P$. Since $H(P)$ is constant w.r.t. model parameters, minimizing cross-entropy is equivalent to minimizing KL divergence during training.

**Properties:**
- KL Divergence is **not symmetric**: $D_{KL}(P \| Q) \neq D_{KL}(Q \| P)$
- Both are always $\geq 0$
- Cross-Entropy $\geq$ Entropy (equality when $P = Q$)
- KL Divergence $= 0$ iff $P = Q$


In [ ]:
import numpy as np
from typing import Union

NDArray = np.ndarray


def cross_entropy(p: NDArray, q: NDArray, eps: float = 1e-12) -> float:
    """Compute cross-entropy H(P, Q) = -sum(P * log(Q)).
    
    Args:
        p: True probability distribution.
        q: Predicted probability distribution.
        eps: Small constant to avoid log(0).
    
    Returns:
        Cross-entropy value.
    """
    q = np.clip(q, eps, 1.0)  # numerical stability
    return -np.sum(p * np.log(q))


def entropy(p: NDArray, eps: float = 1e-12) -> float:
    """Compute Shannon entropy H(P) = -sum(P * log(P))."""
    p = np.clip(p, eps, 1.0)
    return -np.sum(p * np.log(p))


def kl_divergence(p: NDArray, q: NDArray, eps: float = 1e-12) -> float:
    """Compute KL Divergence D_KL(P || Q) = sum(P * log(P/Q)).
    
    Args:
        p: True probability distribution.
        q: Predicted probability distribution.
        eps: Small constant to avoid log(0) and division by zero.
    
    Returns:
        KL divergence value.
    """
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    return np.sum(p * np.log(p / q))


In [ ]:
# --- Example: Compare two distributions ---
p = np.array([0.4, 0.3, 0.2, 0.1])  # true distribution
q = np.array([0.25, 0.25, 0.25, 0.25])  # predicted (uniform)

h_p = entropy(p)
h_pq = cross_entropy(p, q)
dkl = kl_divergence(p, q)

print(f"Entropy H(P):           {h_p:.6f}")
print(f"Cross-Entropy H(P,Q):   {h_pq:.6f}")
print(f"KL Divergence D_KL:     {dkl:.6f}")
print(f"H(P) + D_KL(P||Q):      {h_p + dkl:.6f}")
print(f"\nVerify: H(P,Q) == H(P) + D_KL(P||Q): {np.isclose(h_pq, h_p + dkl)}")

# Asymmetry of KL divergence
print(f"\nD_KL(P||Q) = {kl_divergence(p, q):.6f}")
print(f"D_KL(Q||P) = {kl_divergence(q, p):.6f}")
print("KL Divergence is NOT symmetric!")

In [ ]:
# --- Verify against scipy ---
from scipy.special import rel_entr
from scipy.stats import entropy as sp_entropy

scipy_kl = np.sum(rel_entr(p, q))
scipy_ce = sp_entropy(p, q)

print(f"Our KL Divergence:    {dkl:.6f}")
print(f"Scipy KL Divergence:  {scipy_kl:.6f}")
print(f"Our Cross-Entropy:    {h_pq:.6f}")
print(f"Scipy Cross-Entropy:  {scipy_ce:.6f}")

## Key Takeaways
- **Cross-Entropy = Entropy + KL Divergence** — this is why minimizing CE minimizes KL during training
- KL Divergence is **asymmetric** — direction matters (forward KL vs reverse KL have different behaviors)
- In practice, we use **cross-entropy loss** because it's simpler and equivalent for optimization
- Forward KL ($D_{KL}(P\|Q)$) is **mean-seeking**, reverse KL ($D_{KL}(Q\|P)$) is **mode-seeking**